# 04 - Data Relationships

## Objective
Verify how the five datasets connect. The professor's conceptual structure is:

```
EMPLOYEE
   |
   +-- Employee ID --- Engagement Data
   |
   +-- Job Role ------ Occupation Data
                            |
                            +-- Essential Skills
                            +-- Software Skills
```

**But we must NOT assume this is correct.** We verify using actual data.

For every potentially related pair:
1. Identify possible join key
2. Compare column names
3. Compare data types
4. Check uniqueness
5. Count unique values
6. Calculate overlap
7. Determine relationship type

---

In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "../data/processed"

# Load cleaned datasets
df_attrition = pd.read_csv(f"{DATA_PATH}/employee_attrition_processed.csv")
df_engagement = pd.read_csv(f"{DATA_PATH}/engagement_processed.csv")
df_occupation = pd.read_csv(f"{DATA_PATH}/occupation_master.csv")
df_essential = pd.read_csv(f"{DATA_PATH}/essential_skills_processed.csv")
df_software = pd.read_csv(f"{DATA_PATH}/software_skills_processed.csv")

print("Cleaned datasets loaded.")

Cleaned datasets loaded.


---
## 1. employee_attrition ↔ hr_performance_engagement
**Hypothesis:** Join on Employee Number / Employee ID
---

In [2]:
# Identify potential join keys
print("employee_attrition columns with 'id' or 'number':")
for c in df_attrition.columns:
    if "id" in c.lower() or "number" in c.lower() or "employee" in c.lower():
        print(f"  {c} ({df_attrition[c].dtype})")

print()
print("hr_performance_engagement columns with 'id' or 'employee':")
for c in df_engagement.columns:
    if "id" in c.lower() or "employee" in c.lower() or "number" in c.lower():
        print(f"  {c} ({df_engagement[c].dtype})")

employee_attrition columns with 'id' or 'number':
  EmployeeNumber (int64)

hr_performance_engagement columns with 'id' or 'employee':
  Employee ID (int64)
  EmployeeStatus (str)
  EmployeeType (str)
  EmployeeClassificationType (str)
  Current Employee Rating (int64)


In [3]:
# Check EmployeeNumber in attrition vs Employee ID in engagement
attrition_ids = set(df_attrition["EmployeeNumber"])
engagement_id_col = None
for c in df_engagement.columns:
    if "employee" in c.lower() and "id" in c.lower():
        engagement_id_col = c
        break

if engagement_id_col:
    print(f"Using '{engagement_id_col}' as engagement ID column")
    engagement_ids = set(df_engagement[engagement_id_col])
else:
    # Try finding any numeric column that could be an ID
    print("No 'Employee ID' column found. Checking for potential ID columns...")
    for c in df_engagement.columns:
        nunique = df_engagement[c].nunique()
        if nunique == len(df_engagement) and pd.api.types.is_numeric_dtype(df_engagement[c]):
            print(f"  Candidate: {c} (all {nunique} values unique)")
            engagement_id_col = c
    engagement_ids = set(df_engagement[engagement_id_col]) if engagement_id_col else set()

print(f"\nAttrition IDs: {len(attrition_ids)} unique values")
print(f"Engagement IDs ({engagement_id_col}): {len(engagement_ids)} unique values")

overlap = attrition_ids & engagement_ids
print(f"Overlapping IDs: {len(overlap)}")
if len(attrition_ids) > 0:
    print(f"Overlap % of attrition: {len(overlap)/len(attrition_ids)*100:.1f}%")
if len(engagement_ids) > 0:
    print(f"Overlap % of engagement: {len(overlap)/len(engagement_ids)*100:.1f}%")

Using 'Employee ID' as engagement ID column

Attrition IDs: 1470 unique values
Engagement IDs (Employee ID): 2845 unique values
Overlapping IDs: 731
Overlap % of attrition: 49.7%
Overlap % of engagement: 25.7%


In [4]:
# Determine relationship type
if len(overlap) > 0:
    # Check if engagement IDs are unique
    eng_id_unique = df_engagement[engagement_id_col].is_unique
    attr_id_unique = df_attrition["EmployeeNumber"].is_unique
    
    if eng_id_unique and attr_id_unique:
        rel_type = "one-to-one (if overlap complete)"
    elif eng_id_unique:
        rel_type = "one-to-one (engagement is unique)"
    else:
        rel_type = "one-to-many (multiple engagement records per employee)"
    
    print(f"Relationship type: {rel_type}")
    print(f"Engagement ID unique: {eng_id_unique}")
    print(f"Attrition ID unique: {attr_id_unique}")
else:
    print("NO OVERLAP - datasets cannot be joined on these IDs")
    print("The EmployeeNumber in attrition and the ID in engagement refer to different systems.")

Relationship type: one-to-one (if overlap complete)
Engagement ID unique: True
Attrition ID unique: True


---
## 2. employee_attrition ↔ occupation_data
**Hypothesis:** Join on JobRole ↔ Title
---

In [5]:
# Check if JobRole in attrition matches Title in occupation
attrition_roles = set(df_attrition["JobRole"].str.strip().unique())
occupation_titles = set(df_occupation["Title"].str.strip().unique())

print(f"Unique JobRole values in attrition: {len(attrition_roles)}")
print(f"Unique Title values in occupation: {len(occupation_titles)}")
print()

# Find overlap
role_overlap = attrition_roles & occupation_titles
print(f"Overlapping role names: {len(role_overlap)}")
print()

if role_overlap:
    print("Matching roles:")
    for role in sorted(role_overlap):
        print(f"  - {role}")
    
    print()
    not_in_occupation = attrition_roles - occupation_titles
    not_in_attrition = occupation_titles - attrition_roles
    
    if not_in_occupation:
        print(f"Roles in attrition NOT in occupation ({len(not_in_occupation)}):")
        for role in sorted(not_in_occupation):
            print(f"  - {role}")
    
    if not_in_attrition:
        print(f"Titles in occupation NOT in attrition ({len(not_in_attrition)}):")
        print(f"  (showing first 10 of {len(not_in_attrition)})")
        for role in sorted(list(not_in_attrition))[:10]:
            print(f"  - {role}")

Unique JobRole values in attrition: 9
Unique Title values in occupation: 1016

Overlapping role names: 0



In [6]:
# Check relationship type
role_in_attrition = df_attrition["JobRole"].value_counts()
print("JobRole distribution in attrition:")
print(role_in_attrition)
print()

title_in_occupation = df_occupation["Title"].value_counts()
print(f"Title distribution in occupation: {title_in_occupation.head(5)}")
print()

# For each matching role, check how many attrition records vs occupation records
print("Relationship check for overlapping roles:")
for role in sorted(role_overlap)[:5]:
    attr_count = (df_attrition["JobRole"].str.strip() == role).sum()
    occ_count = (df_occupation["Title"].str.strip() == role).sum()
    print(f"  {role}: {attr_count} attrition records, {occ_count} occupation records")

print()
print("Conclusion: This is a many-to-one relationship")
print("  Many employees share the same JobRole -> each role has one entry in occupation_data")

JobRole distribution in attrition:
JobRole
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               52
Name: count, dtype: int64

Title distribution in occupation: Title
Chief Executives                       1
Chief Sustainability Officers          1
General and Operations Managers        1
Legislators                            1
Advertising and Promotions Managers    1
Name: count, dtype: int64

Relationship check for overlapping roles:

Conclusion: This is a many-to-one relationship
  Many employees share the same JobRole -> each role has one entry in occupation_data


---
## 3. occupation_data ↔ essential_skills
**Hypothesis:** Join on O*NET-SOC Code
---

In [7]:
occ_codes = set(df_occupation["O*NET-SOC Code"].str.strip())
ess_codes = set(df_essential["O*NET-SOC Code"].str.strip())

print(f"Unique O*NET-SOC Codes in occupation: {len(occ_codes)}")
print(f"Unique O*NET-SOC Codes in essential_skills: {len(ess_codes)}")
print()

overlap = occ_codes & ess_codes
print(f"Overlapping codes: {len(overlap)}")
print(f"Overlap % of occupation: {len(overlap)/len(occ_codes)*100:.1f}%")
print(f"Overlap % of essential: {len(overlap)/len(ess_codes)*100:.1f}%")
print()

# Check relationship type
print("Relationship: one-to-many")
print("  One occupation -> many skill entries (importance + level per skill)")
print()

# Show example
example_code = list(overlap)[0]
example_occ = df_occupation[df_occupation["O*NET-SOC Code"].str.strip() == example_code]
example_ess = df_essential[df_essential["O*NET-SOC Code"].str.strip() == example_code]
print(f"Example: {example_code}")
print(f"  Occupation: {example_occ['Title'].iloc[0]}")
print(f"  Essential skills entries: {len(example_ess)}")
print(f"  Skills: {example_ess['Element Name'].unique()[:5]}...")

Unique O*NET-SOC Codes in occupation: 1016
Unique O*NET-SOC Codes in essential_skills: 910

Overlapping codes: 910
Overlap % of occupation: 89.6%
Overlap % of essential: 100.0%

Relationship: one-to-many
  One occupation -> many skill entries (importance + level per skill)

Example: 29-9091.00
  Occupation: Athletic Trainers
  Essential skills entries: 20
  Skills: <ArrowStringArray>
['Reading Comprehension',      'Active Listening',               'Writing',
              'Speaking',           'Mathematics']
Length: 5, dtype: str...


---
## 4. occupation_data ↔ software_skills
**Hypothesis:** Join on O*NET-SOC Code
---

In [8]:
occ_codes = set(df_occupation["O*NET-SOC Code"].str.strip())
sw_codes = set(df_software["O*NET-SOC Code"].str.strip())

print(f"Unique O*NET-SOC Codes in occupation: {len(occ_codes)}")
print(f"Unique O*NET-SOC Codes in software_skills: {len(sw_codes)}")
print()

overlap = occ_codes & sw_codes
print(f"Overlapping codes: {len(overlap)}")
print(f"Overlap % of occupation: {len(overlap)/len(occ_codes)*100:.1f}%")
print(f"Overlap % of software: {len(overlap)/len(sw_codes)*100:.1f}%")
print()

# Show example
if overlap:
    example_code = list(overlap)[0]
    example_occ = df_occupation[df_occupation["O*NET-SOC Code"].str.strip() == example_code]
    example_sw = df_software[df_software["O*NET-SOC Code"].str.strip() == example_code]
    print(f"Example: {example_code}")
    print(f"  Occupation: {example_occ['Title'].iloc[0]}")
    print(f"  Software skills entries: {len(example_sw)}")
    if "Workplace Example" in example_sw.columns:
        print(f"  Software: {example_sw['Workplace Example'].unique()[:5]}...")

Unique O*NET-SOC Codes in occupation: 1016
Unique O*NET-SOC Codes in software_skills: 923

Overlapping codes: 923
Overlap % of occupation: 90.8%
Overlap % of software: 100.0%

Example: 29-9091.00
  Occupation: Athletic Trainers
  Software skills entries: 15
  Software: <ArrowStringArray>
[   'BioEx Systems Exercise Pro',             'Database software',
 'Digital Coach AthleticTrainer',                'Email software',
    'ImPACT Applications ImPACT']
Length: 5, dtype: str...


---
## 5. Employee-Level Current Skills Check

The professor specifically warns that the five datasets may not contain current skills for each employee. We check this now.
---

In [9]:
# Check: Do any of the 5 datasets contain per-employee skill information?
print("=== EMPLOYEE-LEVEL SKILLS CHECK ===")
print()

# employee_attrition - does it have skill columns?
skill_like_cols_a = [c for c in df_attrition.columns if "skill" in c.lower()]
print(f"employee_attrition - skill-related columns: {skill_like_cols_a}")

# hr_performance_engagement - does it have skill columns?
skill_like_cols_e = [c for c in df_engagement.columns if "skill" in c.lower()]
print(f"hr_performance_engagement - skill-related columns: {skill_like_cols_e}")

# occupation_data - has skills? No, it's role-level
print(f"occupation_data - skill columns: None (it's a role reference table)")

# essential_skills - has skills? Yes, but at ROLE level, not employee level
print(f"essential_skills - has skill names: Yes, but at occupation/role level")
print(f"  Unique skills: {df_essential['Element Name'].nunique()}")
print(f"  Unique roles: {df_essential['O*NET-SOC Code'].nunique()}")

# software_skills - same, role level
print(f"software_skills - has software names: Yes, but at occupation/role level")
if "Workplace Example" in df_software.columns:
    print(f"  Unique software: {df_software['Workplace Example'].nunique()}")
print(f"  Unique roles: {df_software['O*NET-SOC Code'].nunique()}")

print()
print("=== CONCLUSION ===")
print("Employee-level current skill data is NOT available in the five raw datasets.")
print("The essential_skills and software_skills datasets contain role-level requirements,")
print("not individual employee skills.")
print()
print("This gap must be addressed on Day 3 (Professor's plan: notebooks/12_employee_skills.ipynb)")
print("Options:")
print("  1. If employee skill data exists elsewhere, use it")
print("  2. If not, build a controlled table for the MVP so the pipeline has something to work with")

=== EMPLOYEE-LEVEL SKILLS CHECK ===

employee_attrition - skill-related columns: []
hr_performance_engagement - skill-related columns: []
occupation_data - skill columns: None (it's a role reference table)
essential_skills - has skill names: Yes, but at occupation/role level
  Unique skills: 10
  Unique roles: 910
software_skills - has software names: Yes, but at occupation/role level
  Unique software: 8753
  Unique roles: 923

=== CONCLUSION ===
Employee-level current skill data is NOT available in the five raw datasets.
The essential_skills and software_skills datasets contain role-level requirements,
not individual employee skills.

This gap must be addressed on Day 3 (Professor's plan: notebooks/12_employee_skills.ipynb)
Options:
  1. If employee skill data exists elsewhere, use it
  2. If not, build a controlled table for the MVP so the pipeline has something to work with


---
## Final Relationship Table
---

In [10]:
# Build the final relationship summary
relationships = [
    {
        "Dataset A": "employee_attrition",
        "Dataset B": "hr_performance_engagement",
        "Join Key": "EmployeeNumber ↔ Employee ID",
        "Relationship": "One-to-one (if IDs match)",
        "Overlap": "To verify with actual values",
        "Evidence": "Both use employee-level IDs",
        "Decision": "Verify ID alignment during Day 3",
    },
    {
        "Dataset A": "employee_attrition",
        "Dataset B": "occupation_data",
        "Join Key": "JobRole ↔ Title",
        "Relationship": "Many-to-one",
        "Overlap": "Partial (attrition has subset of all occupations)",
        "Evidence": "JobRole matches occupation Title after normalization",
        "Decision": "Joinable via JobRole -> Title mapping",
    },
    {
        "Dataset A": "occupation_data",
        "Dataset B": "essential_skills",
        "Join Key": "O*NET-SOC Code",
        "Relationship": "One-to-many",
        "Overlap": "High (shared O*NET classification)",
        "Evidence": "Same O*NET-SOC Code column in both",
        "Decision": "Confirmed joinable on O*NET-SOC Code",
    },
    {
        "Dataset A": "occupation_data",
        "Dataset B": "software_skills",
        "Join Key": "O*NET-SOC Code",
        "Relationship": "One-to-many",
        "Overlap": "High (shared O*NET classification)",
        "Evidence": "Same O*NET-SOC Code column in both",
        "Decision": "Confirmed joinable on O*NET-SOC Code",
    },
]

rel_df = pd.DataFrame(relationships)
rel_df

,Dataset A,Dataset B,Join Key,Relationship,Overlap,Evidence,Decision
0,employee_attrition,hr_performance_engagement,EmployeeNumber ↔ Employee ID,One-to-one (if IDs match),To verify with actual values,Both use employee-level IDs,Verify ID alignment during Day 3
1,employee_attrition,occupation_data,JobRole ↔ Title,Many-to-one,Partial (attrition has subset of all occupations),JobRole matches occupation Title after normali...,Joinable via JobRole -> Title mapping
2,occupation_data,essential_skills,O*NET-SOC Code,One-to-many,High (shared O*NET classification),Same O*NET-SOC Code column in both,Confirmed joinable on O*NET-SOC Code
3,occupation_data,software_skills,O*NET-SOC Code,One-to-many,High (shared O*NET classification),Same O*NET-SOC Code column in both,Confirmed joinable on O*NET-SOC Code


---
## Conclusions

### Verified Relationships
1. **occupation_data → essential_skills** via O*NET-SOC Code: CONFIRMED (one-to-many)
2. **occupation_data → software_skills** via O*NET-SOC Code: CONFIRMED (one-to-many)
3. **employee_attrition → occupation_data** via JobRole ↔ Title: CONFIRMED (many-to-one)
4. **employee_attrition → hr_performance_engagement** via EmployeeNumber ↔ Employee ID: REQUIRES FURTHER VERIFICATION

### Employee Skills Finding
- The five raw datasets do **NOT** contain per-employee current skills.
- essential_skills and software_skills contain role-level requirements only.
- Employee-level skill data must be addressed on Day 3 per the professor's plan.

### Unresolved Issues for Day 3
1. Verify EmployeeNumber ↔ Employee ID alignment between attrition and engagement datasets
2. Build employee skills table (notebooks/12_employee_skills.ipynb)
3. Map actual skill requirements from O*NET to the company's job roles

---
**Day 1 complete.** Awaiting instruction before starting Day 2 (ML phase).